# Serie A — Entrenamiento del Modelo

**Objetivo:** Cargar datos, construir features, entrenar XGBoost y guardar `modelo_italian.pkl`

```
Primera vez     →  Correr celdas 1 a 9 en orden
Reentrenamiento →  Correr solo celda 10
```

**Output:** `modelo_italian.pkl` — usado por `italian_league_app.ipynb`

**Fuente de datos:** football-data.co.uk · código `I1` (Serie A)

In [1]:
# ─────────────────────────────────────────────
# CELDA 1 — Imports
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import joblib
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

print('Imports OK')

Imports OK


In [2]:
# ─────────────────────────────────────────────
# CELDA 2 — Cargar datos
# I1 = Serie A italiana
# Agrega temporadas aquí cuando haya nuevas
# ─────────────────────────────────────────────
temporadas     = ['2021', '2122', '2223', '2324', '2425', '2526']
temporada_test = '2526'
base_url       = 'https://www.football-data.co.uk/mmz4281/{}/I1.csv'

dfs = []
for t in temporadas:
    df_temp = pd.read_csv(base_url.format(t))
    df_temp['season'] = t
    dfs.append(df_temp)

df_raw = pd.concat(dfs, ignore_index=True)
cols   = ['season', 'Date', 'HomeTeam', 'AwayTeam',
          'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']

df = df_raw[cols].dropna(subset=['FTR'])
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

print(f'Total partidos: {len(df)}')
print(f'Rango fechas:   {df["Date"].min().date()} -> {df["Date"].max().date()}')
print(f'\nDistribucion resultados:')
print(df['FTR'].value_counts())
print(f'\nEquipos en el dataset:')
print(sorted(df['HomeTeam'].unique()))

Total partidos: 2210
Rango fechas:   2020-09-19 -> 2026-04-06

Distribucion resultados:
FTR
H    897
A    717
D    596
Name: count, dtype: int64

Equipos en el dataset:
['Atalanta', 'Benevento', 'Bologna', 'Cagliari', 'Como', 'Cremonese', 'Crotone', 'Empoli', 'Fiorentina', 'Frosinone', 'Genoa', 'Inter', 'Juventus', 'Lazio', 'Lecce', 'Milan', 'Monza', 'Napoli', 'Parma', 'Pisa', 'Roma', 'Salernitana', 'Sampdoria', 'Sassuolo', 'Spezia', 'Torino', 'Udinese', 'Venezia', 'Verona']


In [3]:
# ─────────────────────────────────────────────
# CELDA 3 — Funciones de features
# ─────────────────────────────────────────────

def get_team_stats_v2(team, date, df, n_recent=5):
    home_all  = df[(df['HomeTeam'] == team) & (df['Date'] < date)]
    away_all  = df[(df['AwayTeam'] == team) & (df['Date'] < date)]
    all_games = pd.concat([home_all, away_all]).sort_values('Date')
    if len(all_games) < 3:
        return None
    current_season = df[df['Date'] < date]['season'].iloc[-1]
    season_games   = pd.concat([
        home_all[home_all['season'] == current_season],
        away_all[away_all['season'] == current_season]
    ]).sort_values('Date')
    recent = all_games.tail(n_recent)

    def calc_stats(games, team):
        if len(games) == 0: return None
        gf=gc=sot_f=sot_c=wins=draws=home_wins=home_games=0
        for _, r in games.iterrows():
            ih = r['HomeTeam'] == team
            gf    += r['FTHG'] if ih else r['FTAG']
            gc    += r['FTAG'] if ih else r['FTHG']
            sot_f += r['HST']  if ih else r['AST']
            sot_c += r['AST']  if ih else r['HST']
            won  = (ih and r['FTR']=='H') or (not ih and r['FTR']=='A')
            drew = r['FTR'] == 'D'
            if won:  wins  += 1
            if drew: draws += 1
            if ih:
                home_games += 1
                if r['FTR'] == 'H': home_wins += 1
        m = len(games)
        return {'gf_pg': gf/m, 'gc_pg': gc/m, 'dif_goles': (gf-gc)/m,
                'sot_f_pg': sot_f/m, 'sot_c_pg': sot_c/m,
                'win_rate': wins/m, 'draw_rate': draws/m,
                'home_wr': home_wins/home_games if home_games > 0 else wins/m}

    rs = calc_stats(recent, team)
    ss = calc_stats(season_games, team) if len(season_games) >= 3 else rs
    if rs is None: return None
    return {f'recent_{k}': rs[k] for k in rs} | {f'season_{k}': ss[k] if ss else rs[k] for k in rs}


def get_h2h_stats(home_team, away_team, date, df, n=10):
    h2h = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ][lambda x: x['Date'] < date].tail(n)
    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}
    hw = ((h2h['HomeTeam']==home_team)&(h2h['FTR']=='H')).sum() + \
         ((h2h['AwayTeam']==home_team)&(h2h['FTR']=='A')).sum()
    dr = (h2h['FTR']=='D').sum()
    m  = len(h2h)
    return {'h2h_home_wr': hw/m, 'h2h_draw_rate': dr/m, 'h2h_away_wr': (m-hw-dr)/m}


def get_tabla_posicion(team, date, df):
    season = df[df['Date'] < date]['season'].iloc[-1]
    ps     = df[(df['season'] == season) & (df['Date'] < date)]
    tabla  = []
    for eq in pd.concat([ps['HomeTeam'], ps['AwayTeam']]).unique():
        hp = ps[ps['HomeTeam']==eq]; ap = ps[ps['AwayTeam']==eq]
        pts = (hp['FTR']=='H').sum()*3+(hp['FTR']=='D').sum() + \
              (ap['FTR']=='A').sum()*3+(ap['FTR']=='D').sum()
        tabla.append({'equipo': eq, 'pts': pts,
                      'gf': hp['FTHG'].sum()+ap['FTAG'].sum(),
                      'gc': hp['FTAG'].sum()+ap['FTHG'].sum(),
                      'pj': len(hp)+len(ap)})
    tdf = pd.DataFrame(tabla).sort_values('pts', ascending=False).reset_index(drop=True)
    tdf['pos'] = tdf.index + 1
    f = tdf[tdf['equipo']==team]
    if len(f)==0: return {'posicion': 10, 'pts_pg': 1.0, 'dif_goles_szn': 0, 'pct_posicion': 0.5}
    f = f.iloc[0]; pj = max(f['pj'], 1)
    return {'posicion': f['pos'], 'pts_pg': f['pts']/pj,
            'dif_goles_szn': (f['gf']-f['gc'])/pj,
            'pct_posicion': 1-(f['pos']/len(tdf))}


print('Funciones definidas OK')

Funciones definidas OK


In [4]:
# ─────────────────────────────────────────────
# CELDA 4 — Construir dataset
# Tarda ~3-5 minutos
# ─────────────────────────────────────────────

def construir_dataset(df):
    rows = []
    for _, p in df.iterrows():
        ht, at, dt, sn = p['HomeTeam'], p['AwayTeam'], p['Date'], p['season']
        hs  = get_team_stats_v2(ht, dt, df)
        as_ = get_team_stats_v2(at, dt, df)
        if hs is None or as_ is None: continue
        h2h = get_h2h_stats(ht, at, dt, df)
        ht2 = get_tabla_posicion(ht, dt, df)
        at2 = get_tabla_posicion(at, dt, df)
        row = {'date': dt, 'season': sn, 'home_team': ht, 'away_team': at, 'resultado': p['FTR']}
        for k, v in hs.items():  row[f'h_{k}'] = v
        for k, v in as_.items(): row[f'a_{k}'] = v
        for k, v in h2h.items(): row[k] = v
        for k, v in ht2.items(): row[f'h_tabla_{k}'] = v
        for k, v in at2.items(): row[f'a_tabla_{k}'] = v
        row['dif_recent_wr']  = hs['recent_win_rate']  - as_['recent_win_rate']
        row['dif_season_wr']  = hs['season_win_rate']  - as_['season_win_rate']
        row['dif_recent_gf']  = hs['recent_gf_pg']     - as_['recent_gf_pg']
        row['dif_recent_gc']  = hs['recent_gc_pg']     - as_['recent_gc_pg']
        row['dif_recent_dif'] = hs['recent_dif_goles'] - as_['recent_dif_goles']
        row['dif_season_dif'] = hs['season_dif_goles'] - as_['season_dif_goles']
        row['home_advantage'] = hs['recent_home_wr']   - as_['recent_win_rate']
        row['dif_pts_pg']     = ht2['pts_pg']          - at2['pts_pg']
        row['dif_posicion']   = at2['posicion']        - ht2['posicion']
        rows.append(row)
    return pd.DataFrame(rows)


df_v3 = construir_dataset(df)
print(f'Partidos: {len(df_v3)}')
print(f'Features: {len(df_v3.columns) - 5}')
print(df_v3['season'].value_counts().sort_index())

Partidos: 2152
Features: 52
season
2021    348
2122    372
2223    371
2324    377
2425    377
2526    307
Name: count, dtype: int64


In [5]:
# ─────────────────────────────────────────────
# CELDA 5 — Train/Test split temporal
# ─────────────────────────────────────────────
TEMPORADA_TEST = '2526'  # actualizar cuando llegue nueva temporada

train_v3 = df_v3[df_v3['season'] != TEMPORADA_TEST].copy()
test_v3  = df_v3[df_v3['season'] == TEMPORADA_TEST].copy()

feature_cols_v3 = [c for c in df_v3.columns
                   if c not in ['date','season','home_team','away_team','resultado']]

X_train_v3 = train_v3[feature_cols_v3]
X_test_v3  = test_v3[feature_cols_v3]

le = LabelEncoder()
y_train_v3 = le.fit_transform(train_v3['resultado'])
y_test_v3  = le.transform(test_v3['resultado'])

print(f'Train: {len(train_v3)} partidos')
print(f'Test:  {len(test_v3)} partidos (temporada {TEMPORADA_TEST})')
print(f'Features: {len(feature_cols_v3)}')

Train: 1845 partidos
Test:  307 partidos (temporada 2526)
Features: 52


In [6]:
# ─────────────────────────────────────────────
# CELDA 6 — Entrenar XGBoost + Calibrar
# ─────────────────────────────────────────────
xgb_v3 = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    random_state=42, eval_metric='mlogloss', verbosity=0
)
xgb_v3.fit(train_v3[feature_cols_v3], y_train_v3)

model_v3 = CalibratedClassifierCV(xgb_v3, cv=5, method='isotonic')
model_v3.fit(train_v3[feature_cols_v3], y_train_v3)

print('Modelo entrenado y calibrado OK')

Modelo entrenado y calibrado OK


In [7]:
# ─────────────────────────────────────────────
# CELDA 7 — Evaluar modelo
# ─────────────────────────────────────────────
y_pred       = model_v3.predict(test_v3[feature_cols_v3])
y_pred_proba = model_v3.predict_proba(test_v3[feature_cols_v3])

accuracy_nuevo = accuracy_score(y_test_v3, y_pred)
logloss_nuevo  = log_loss(y_test_v3, y_pred_proba)

proba_df = pd.DataFrame(y_pred_proba, columns=['prob_A','prob_D','prob_H'])
proba_df['confianza'] = proba_df[['prob_A','prob_D','prob_H']].max(axis=1)
proba_df['pred']      = le.inverse_transform(y_pred)
proba_df['real']      = le.inverse_transform(y_test_v3)
proba_df['correcto']  = proba_df['pred'] == proba_df['real']

alta_conf     = proba_df[proba_df['confianza'] > 0.55]
acc_alta_conf = alta_conf['correcto'].mean() if len(alta_conf) > 0 else 0

# Baseline: siempre predecir local (H)
baseline = (test_v3['resultado'] == 'H').mean()

print(f'Accuracy general:        {accuracy_nuevo:.1%}')
print(f'Accuracy alta confianza: {acc_alta_conf:.1%} ({len(alta_conf)} partidos >55%)')
print(f'Log Loss:                {logloss_nuevo:.4f}')
print(f'Baseline (siempre H):    {baseline:.1%}')
print(f'Mejora vs baseline:      {(accuracy_nuevo-baseline):+.1%}')
print(f'\nReporte por clase:')
print(classification_report(y_test_v3, y_pred, target_names=le.classes_))

bins   = [0.33, 0.40, 0.45, 0.50, 0.55, 1.0]
labels = ['33-40%','40-45%','45-50%','50-55%','>55%']
proba_df['conf_bin'] = pd.cut(proba_df['confianza'], bins=bins, labels=labels)
print('\nAccuracy por nivel de confianza:')
print(proba_df.groupby('conf_bin', observed=True).agg(
    partidos=('correcto','count'), accuracy=('correcto','mean')
).round(3))

Accuracy general:        51.1%
Accuracy alta confianza: 66.2% (65 partidos >55%)
Log Loss:                1.0101
Baseline (siempre H):    40.1%
Mejora vs baseline:      +11.1%

Reporte por clase:
              precision    recall  f1-score   support

           A       0.54      0.59      0.56       104
           D       0.37      0.09      0.14        80
           H       0.51      0.72      0.60       123

    accuracy                           0.51       307
   macro avg       0.47      0.47      0.43       307
weighted avg       0.48      0.51      0.47       307


Accuracy por nivel de confianza:
          partidos  accuracy
conf_bin                    
33-40%          65     0.338
40-45%          73     0.534
45-50%          63     0.429
50-55%          41     0.634
>55%            65     0.662


In [8]:
# ─────────────────────────────────────────────
# CELDA 8 — Guardar modelo con versionado por fecha
# Autosuficiente — no depende de celdas anteriores
# ─────────────────────────────────────────────
import os
from datetime import datetime

y_pred_v3       = model_v3.predict(X_test_v3)
y_pred_proba_v3 = model_v3.predict_proba(X_test_v3)

accuracy_v3 = accuracy_score(y_test_v3, y_pred_v3)
logloss_v3  = log_loss(y_test_v3, y_pred_proba_v3)

proba_df = pd.DataFrame(y_pred_proba_v3, columns=['prob_A', 'prob_D', 'prob_H'])
proba_df['confianza'] = proba_df.max(axis=1)
proba_df['pred']      = le.inverse_transform(y_pred_v3)
proba_df['real']      = le.inverse_transform(y_test_v3)
proba_df['correcto']  = proba_df['pred'] == proba_df['real']

mask_alta               = proba_df['confianza'] > 0.55
accuracy_alta_confianza = proba_df[mask_alta]['correcto'].mean()
n_alta_confianza        = mask_alta.sum()

os.makedirs('versiones', exist_ok=True)

fecha_hoy    = datetime.now().strftime('%Y-%m-%d')
path_actual  = 'modelo_italian.pkl'
path_version = f'versiones/modelo_{fecha_hoy}.pkl'

payload = {
    'model_v3':                model_v3,
    'feature_cols_v3':         feature_cols_v3,
    'le':                      le,
    'df':                      df,
    'accuracy':                accuracy_v3,
    'accuracy_alta_confianza': accuracy_alta_confianza,
    'logloss':                 logloss_v3,
    'n_alta_confianza':        int(n_alta_confianza),
    'fecha_entrenamiento':     fecha_hoy,
    'temporadas':              temporadas,
    'temporada_test':          temporada_test,
    'partidos_train':          len(X_train_v3),
    'partidos_test':           len(X_test_v3),
}

joblib.dump(payload, path_version)
joblib.dump(payload, path_actual)

print(f'Modelo guardado:')
print(f'  Principal:  {path_actual}')
print(f'  Versión:    {path_version}')
print(f'  Accuracy:   {accuracy_v3:.1%}')
print(f'  Alta conf:  {accuracy_alta_confianza:.1%}')
print(f'  Fecha:      {fecha_hoy}')
print(f'  Train:      {len(X_train_v3)} partidos')

Modelo guardado:
  Principal:  modelo_italian.pkl
  Versión:    versiones/modelo_2026-04-07.pkl
  Accuracy:   51.1%
  Alta conf:  66.2%
  Fecha:      2026-04-07
  Train:      1845 partidos


In [9]:
# ─────────────────────────────────────────────
# CELDA 9 — Verificar modelo guardado
# ─────────────────────────────────────────────
from datetime import datetime
fecha_hoy = datetime.now().strftime('%Y-%m-%d')
datos_guardados = joblib.load(f'versiones/modelo_{fecha_hoy}.pkl')

print('Verificacion del modelo guardado:')
print(f'  Fecha entrenamiento:   {datos_guardados["fecha_entrenamiento"]}')
print(f'  Accuracy:              {datos_guardados["accuracy"]:.1%}')
print(f'  Accuracy alta conf:    {datos_guardados["accuracy_alta_confianza"]:.1%}')
print(f'  Log Loss:              {datos_guardados["logloss"]}')
print(f'  Partidos train:        {datos_guardados["partidos_train"]}')
print(f'  Partidos test:         {datos_guardados["partidos_test"]}')
print(f'  Temporada test:        {datos_guardados["temporada_test"]}')
print(f'  Features:              {len(datos_guardados["feature_cols_v3"])}')

Verificacion del modelo guardado:
  Fecha entrenamiento:   2026-04-07
  Accuracy:              51.1%
  Accuracy alta conf:    66.2%
  Log Loss:              1.0101142296343928
  Partidos train:        1845
  Partidos test:         307
  Temporada test:        2526
  Features:              52


---
## Reentrenamiento

Correr **celda 10** cada 3 jornadas (~30 partidos nuevos disponibles en el CSV).

**Lógica de comparación:**
- Carga el accuracy del modelo anterior desde el pkl
- Entrena un nuevo modelo con los datos frescos
- Solo reemplaza el pkl si `accuracy_nuevo >= accuracy_anterior - 0.02`
- El umbral de -0.02 (2%) tolera varianza natural del test set

In [10]:
# ─────────────────────────────────────────────
# CELDA 10 — Reentrenamiento automatico
# Correr cada 3 jornadas (~30 partidos nuevos)
# ─────────────────────────────────────────────

def reentrenar(temporada_test='2526', umbral_degradacion=0.02):
    """
    Reentrena con datos frescos y reemplaza el pkl
    solo si el accuracy no degrada mas del umbral.

    Comparacion:
        accuracy_nuevo >= accuracy_anterior - umbral_degradacion
        Si True  -> reemplaza modelo_italian.pkl
        Si False -> mantiene el modelo anterior e informa
    """

    # Paso 1: Cargar métricas del modelo anterior
    try:
        ant = joblib.load('modelo_italian.pkl')
        acc_ant  = ant['accuracy']
        accf_ant = ant['accuracy_alta_confianza']
        n_ant    = ant['partidos_train']
        print(f'Modelo anterior:')
        print(f'  Fecha:             {ant["fecha_entrenamiento"]}')
        print(f'  Accuracy:          {acc_ant:.1%}')
        print(f'  Accuracy alta conf:{accf_ant:.1%}')
        print(f'  Partidos train:    {n_ant}')
    except FileNotFoundError:
        print('No hay modelo anterior — entrenando desde cero')
        acc_ant = accf_ant = 0
        n_ant   = 0

    # Paso 2: Verificar partidos nuevos disponibles
    dfs = []
    for t in ['2021','2122','2223','2324','2425','2526']:
        df_temp = pd.read_csv(f'https://www.football-data.co.uk/mmz4281/{t}/I1.csv')
        df_temp['season'] = t
        dfs.append(df_temp)

    df_new = pd.concat(dfs, ignore_index=True)
    cols   = ['season','Date','HomeTeam','AwayTeam','FTHG','FTAG','FTR','HS','AS','HST','AST','HC','AC']
    df_new = df_new[cols].dropna(subset=['FTR'])
    df_new['Date'] = pd.to_datetime(df_new['Date'], dayfirst=True)
    df_new = df_new.sort_values('Date').reset_index(drop=True)

    n_nuevos = len(df_new[df_new['season'] != temporada_test]) - n_ant
    print(f'\nPartidos nuevos disponibles: {n_nuevos}')

    if n_nuevos < 20 and n_ant > 0:
        print(f'Menos de 20 partidos nuevos — esperar hasta tener 30+')
        print('Para forzar igual, cambia el umbral a < 0')
        return

    # Paso 3: Reconstruir dataset con datos frescos
    print('\nReconstruyendo dataset (~5 min)...')
    df_v3_nuevo = construir_dataset(df_new)

    # Paso 4: Entrenar nuevo modelo
    tr = df_v3_nuevo[df_v3_nuevo['season'] != temporada_test].copy()
    te = df_v3_nuevo[df_v3_nuevo['season'] == temporada_test].copy()
    fc = [c for c in df_v3_nuevo.columns
          if c not in ['date','season','home_team','away_team','resultado']]

    le_n = LabelEncoder()
    y_tr = le_n.fit_transform(tr['resultado'])
    y_te = le_n.transform(te['resultado'])

    xgb_n = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                          random_state=42, eval_metric='mlogloss', verbosity=0)
    xgb_n.fit(tr[fc], y_tr)

    mdl_n = CalibratedClassifierCV(xgb_n, cv=5, method='isotonic')
    mdl_n.fit(tr[fc], y_tr)

    # Paso 5: Evaluar nuevo modelo
    y_p  = mdl_n.predict(te[fc])
    y_pp = mdl_n.predict_proba(te[fc])

    acc_nuevo  = accuracy_score(y_te, y_p)
    ll_nuevo   = log_loss(y_te, y_pp)

    pdf = pd.DataFrame(y_pp, columns=['p_A','p_D','p_H'])
    pdf['conf'] = pdf[['p_A','p_D','p_H']].max(axis=1)
    pdf['pred'] = le_n.inverse_transform(y_p)
    pdf['real'] = le_n.inverse_transform(y_te)
    pdf['ok']   = pdf['pred'] == pdf['real']
    accf_nuevo  = pdf[pdf['conf']>0.55]['ok'].mean() if (pdf['conf']>0.55).sum()>0 else 0

    # Paso 6: Comparar y decidir
    print(f'\nComparacion:')
    print(f'  Metrica              Anterior    Nuevo       Cambio')
    print(f'  ────────────────────────────────────────────────────')
    print(f'  Accuracy             {acc_ant:.1%}      {acc_nuevo:.1%}      {(acc_nuevo-acc_ant):+.1%}')
    print(f'  Accuracy alta conf   {accf_ant:.1%}      {accf_nuevo:.1%}      {(accf_nuevo-accf_ant):+.1%}')
    print(f'  Partidos train       {n_ant}       {len(tr)}')

    if acc_nuevo >= acc_ant - umbral_degradacion:
        fecha_hoy = datetime.now().strftime('%Y-%m-%d')
        payload_nuevo = {
            'model_v3':                mdl_n,
            'feature_cols_v3':         fc,
            'le':                      le_n,
            'df':                      df_new,
            'accuracy':                round(acc_nuevo, 4),
            'accuracy_alta_confianza': round(accf_nuevo, 4),
            'logloss':                 round(ll_nuevo, 4),
            'n_alta_confianza':        int((pdf['conf']>0.55).sum()),
            'fecha_entrenamiento':     fecha_hoy,
            'temporadas':              ['2021','2122','2223','2324','2425','2526'],
            'temporada_test':          temporada_test,
            'partidos_train':          len(tr),
            'partidos_test':           len(te),
        }
        joblib.dump(payload_nuevo, f'versiones/modelo_{fecha_hoy}.pkl')
        joblib.dump(payload_nuevo, 'modelo_italian.pkl')
        print(f'\n  RESULTADO: Modelo ACTUALIZADO')
    else:
        print(f'\n  RESULTADO: Modelo NO actualizado')
        print(f'  Accuracy bajo {umbral_degradacion:.0%} del anterior — revisar datos')


# Ejecutar
reentrenar(temporada_test='2526', umbral_degradacion=0.02)

Modelo anterior:
  Fecha:             2026-04-07
  Accuracy:          51.1%
  Accuracy alta conf:66.2%
  Partidos train:    1845

Partidos nuevos disponibles: 55

Reconstruyendo dataset (~5 min)...

Comparacion:
  Metrica              Anterior    Nuevo       Cambio
  ────────────────────────────────────────────────────
  Accuracy             51.1%      51.1%      +0.0%
  Accuracy alta conf   66.2%      66.2%      +0.0%
  Partidos train       1845       1845

  RESULTADO: Modelo ACTUALIZADO


In [11]:
# ─────────────────────────────────────────────
# CELDA 11 — Ver historial de versiones
# Correr en cualquier momento
# ─────────────────────────────────────────────
import os

versiones_dir = 'versiones'
if not os.path.exists(versiones_dir):
    print('No hay versiones guardadas aún')
else:
    archivos = sorted(os.listdir(versiones_dir))
    if not archivos:
        print('No hay versiones guardadas aún')
    else:
        print(f'{"Versión":<30} {"Fecha":<15} {"Accuracy":<12} {"Alta conf":<12} {"Partidos train"}')
        print('─' * 85)
        for archivo in archivos:
            path = os.path.join(versiones_dir, archivo)
            try:
                v = joblib.load(path)
                print(
                    f'{archivo:<30} '
                    f'{v.get("fecha_entrenamiento", "?"):<15} '
                    f'{v.get("accuracy", 0):<12.1%} '
                    f'{v.get("accuracy_alta_confianza", 0):<12.1%} '
                    f'{v.get("partidos_train", 0)}'
                )
            except Exception as e:
                print(f'{archivo:<30} Error al leer: {e}')

Versión                        Fecha           Accuracy     Alta conf    Partidos train
─────────────────────────────────────────────────────────────────────────────────────
modelo_2026-04-07.pkl          2026-04-07      51.1%        66.1%        1845
